In [ ]:
import os, pandas as pd

pds = '/kaggle/input/datasets/nina2025/ps-s6e6/'
sample_submission = '/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv'

In [ ]:
ddf = [
    {'subm':f'0.96652.csv','short':f'652'}, # 0
    {'subm':f'0.96707.csv','short':f'707'}, # 1
    {'subm':f'0.96722.csv','short':f'722'}, # 2
    {'subm':f'0.96751.csv','short':f'751'}, # 3
    {'subm':f'0.96771.csv','short':f'771'}, # 4
    {'subm':f'0.96807.csv','short':f'807'}, # 5
    {'subm':f'0.96889.csv','short':f'889'}, # 6
    {'subm':f'0.96918.csv','short':f'918'}, # 7
    {'subm':f'0.96920.csv','short':f'920'}, # 8
    {'subm':f'0.96933.csv','short':f'933'}, # 9
    {'subm':f'0.96973.csv','short':f'973'}, # 10
    {'subm':f'0.96983.csv','short':f'983'}, # 11
    {'subm':f'0.97007.csv','short':f'1007'},# 12
    {'subm':f'0.96936.csv','short':f'936'}, # 13
]

In [ ]:
I = [3,4,5,7,10]  # LB=0.96996
I = [3,4,5,9,10]  # LB=0.96997
I = [6,7,8,9,10]  # LB=0.96955
I = [3,4,6,8,10]  # LB=0.96994
I = [2,4,5,9,10]  # LB=0.96929

I = [  4,5,9,10]  # LB=0.96927
I = [  3,5,8,10]  # LB=0.96993
I = [  3,5,9,10]  # LB=0.96991
I = [  3,5,8, 9]  # LB=0.96935

I = [5,8,10,11,12]# LB=0.97009

# ----------------------------
#       2-level voting:
# ----------------------------
I =[
    [   4,5,9,10], # LB=0.96927
    [   3,5,8, 9], # LB=0.96935
    [   3,5,8,10], # LB=0.96993    
    [ 3,4,6,8,10], # LB=0.96994  v.14.LB = 0.96997
    [ 3,4,5,7,10], # LB=0.96996
    [ 3,4,5,9,10], # LB=0.96997
]
# ----------------------------
I =[
    [   4,5,9,10], # LB=0.96927
    [  13,5,8, 9], # LB=?
    [   3,5,8,10], # LB=0.96993    
    [ 3,4,6,8,10], # LB=0.96994  v.15.LB = 0.97007
    [13,4,5,7,10], # LB=?
    [ 5,8,10,11,12]# LB=0.97009
]
# -------------------------------------------------

_ =  [1,2,3,4,5,6,7,8,9,10,11,12,13]
#------------------------------------
I = [[      4,5,  7,  9,   11      ], #lb=?
#------------------------------------
     [          6,  8,9,10,   12   ], #lb=?
#------------------------------------
     [    3,  5,    8,     11,   13], #lb=? 
#------------------------------------                v.17.LB = ?
     [      4,  6,  8,  10,11      ], #lb=?
#------------------------------------
     [  2,    5,  7,    10,      13], #lb=?
#------------------------------------
     [1,  3,4,    7,             13], #lb=?
#------------------------------------
     [        5,    8,  10,11,12,  ]] #LB=0.97009
#------------------------------------

In [ ]:
# if len(I)==4: dfs=vote4()

def vote4(ddf=ddf,I=I):
    subms  = [pd.read_csv(pds+f['subm']).rename(columns={'class':f['short']}) for f in ddf]
    shorts = [                                                   f['short']   for f in ddf]
    voters = [{'subm':subms[i],'short':shorts[i]} for i in I]
    
    dfs    = pd.merge(pd.merge(pd.merge(subms[I[0]],
                                        subms[I[1]],on='id'),
                                        subms[I[2]],on='id'),
                                        subms[I[3]],on='id')

    def comp(x):
        _mask = x[voters[0]['short']]
        rezult = all([x[vtr['short']]==_mask for vtr in voters])
        return '=' if rezult==True else '!='
    
    dfs['comp'] = dfs.apply(lambda x: comp(x), axis=1)
    #---------------------------------------------------
    display(dfs.head(4)) ;print(dfs['comp'].value_counts(),'\n')
    #-----------------------------------------------------
    
    def vote(x):
        sh = [vtr['short'] for vtr in voters]
        if x[sh[0]]==x[sh[1]]==x[sh[2]]          : return x[sh[2]]
        if x[sh[0]]==x[sh[1]]          ==x[sh[3]]: return x[sh[3]]
        if x[sh[0]]          ==x[sh[2]]==x[sh[3]]: return x[sh[3]]
        if           x[sh[1]]==x[sh[2]]==x[sh[3]]: return x[sh[3]]
        return x[sh[3]]
    
    dfs['vote'] = dfs.apply(lambda x: vote(x), axis=1)
    #----------------------------------------------------
    display(dfs.head(4)) ;print(dfs['vote'].value_counts(),'\n')
    #------------------------------------------------------
    
    def vs(x,participant): return '=' if x['vote']==x[participant] else '!='
    
    sh = [vtr['short'] for vtr in voters]
    
    dfs[f'vs {sh[0]}'] = dfs.apply(lambda x:vs(x,sh[0]),axis=1) 
    dfs[f'vs {sh[1]}'] = dfs.apply(lambda x:vs(x,sh[1]),axis=1)
    dfs[f'vs {sh[2]}'] = dfs.apply(lambda x:vs(x,sh[2]),axis=1)
    dfs[f'vs {sh[3]}'] = dfs.apply(lambda x:vs(x,sh[3]),axis=1)
    #----------------------------------------------------
    display(dfs.head(4)) 
    #------------------------------------------------------
    print(dfs[f'vs {sh[0]}'].value_counts(),'\n')
    print(dfs[f'vs {sh[1]}'].value_counts(),'\n')
    print(dfs[f'vs {sh[2]}'].value_counts(),'\n')
    print(dfs[f'vs {sh[3]}'].value_counts(),'\n')
    return dfs


if len(I) == 4: dfs = vote4()

In [ ]:
#if len(I)==5: dfs=vote5()

def vote5(ddf=ddf,I=I):

    subms  = [pd.read_csv(pds+f['subm']).rename(columns={'class':f['short']}) for f in ddf]
    shorts = [                                                   f['short']   for f in ddf]
    voters = [{'subm':subms[i],'short':shorts[i]} for i in I]
    
    dfs    = pd.merge(pd.merge(pd.merge(pd.merge(subms[I[0]],
                                                 subms[I[1]],on='id'),
                                                 subms[I[2]],on='id'),
                                                 subms[I[3]],on='id'),
                                                 subms[I[4]],on='id')
    def comp(x):
        _mask = x[voters[0]['short']]
        rezult = all([x[vtr['short']]==_mask for vtr in voters])
        return '=' if rezult==True else '!='
    
    dfs['comp'] = dfs.apply(lambda x: comp(x), axis=1)
    #---------------------------------------------------
    display(dfs.head(5)) ;print(dfs['comp'].value_counts(),'\n')
    #-----------------------------------------------------
    
    def vote(x):
        sh = [vtr['short'] for vtr in voters]
        if x[sh[0]]==x[sh[1]]==x[sh[2]]==x[sh[3]]          : return x[sh[3]]
        if x[sh[0]]==x[sh[1]]==x[sh[2]]          ==x[sh[4]]: return x[sh[4]]
        if x[sh[0]]==x[sh[1]]          ==x[sh[3]]==x[sh[4]]: return x[sh[4]]
        if x[sh[0]]          ==x[sh[2]]==x[sh[3]]==x[sh[4]]: return x[sh[4]]
        if           x[sh[1]]==x[sh[2]]==x[sh[3]]==x[sh[4]]: return x[sh[4]]
        return x[sh[4]]
    
    dfs['vote'] = dfs.apply(lambda x: vote(x), axis=1)
    #----------------------------------------------------
    display(dfs.head(5)) ;print(dfs['vote'].value_counts(),'\n')
    #------------------------------------------------------
    
    def vs(x,participant): return '=' if x['vote']==x[participant] else '!='
    
    sh = [vtr['short'] for vtr in voters]
    
    dfs[f'vs {sh[0]}'] = dfs.apply(lambda x:vs(x,sh[0]),axis=1) 
    dfs[f'vs {sh[1]}'] = dfs.apply(lambda x:vs(x,sh[1]),axis=1)
    dfs[f'vs {sh[2]}'] = dfs.apply(lambda x:vs(x,sh[2]),axis=1)
    dfs[f'vs {sh[3]}'] = dfs.apply(lambda x:vs(x,sh[3]),axis=1)
    dfs[f'vs {sh[4]}'] = dfs.apply(lambda x:vs(x,sh[4]),axis=1) 
    #----------------------------------------------------
    display(dfs.head(5)) 
    #------------------------------------------------------
    print(dfs[f'vs {sh[0]}'].value_counts(),'\n')
    print(dfs[f'vs {sh[1]}'].value_counts(),'\n')
    print(dfs[f'vs {sh[2]}'].value_counts(),'\n')
    print(dfs[f'vs {sh[3]}'].value_counts(),'\n')
    print(dfs[f'vs {sh[4]}'].value_counts(),'\n')
    return dfs

if len(I) == 5: dfs = vote5()

In [ ]:
if len(I) == 6:
    df  = pd.read_csv(sample_submission)

    def save(df,name_csv):
        df.rename(columns={'vote':'class'},inplace=True)
        df[['id','class']].to_csv(name_csv,index=False)

    df40 = vote4(I=I[0]) ;save(df40,'a4.csv')
    df41 = vote4(I=I[1]) ;save(df41,'b4.csv')
    df42 = vote4(I=I[2]) ;save(df42,'c4.csv')
    df50 = vote5(I=I[3]) ;save(df50,'d5.csv')
    df51 = vote5(I=I[4]) ;save(df51,'e5.csv')
    df52 = vote5(I=I[5]) ;save(df52,'f5.csv')

    pds = '/kaggle/working/'

    ddf = [
        {'subm':f'a4.csv','short':f'a4'}, # 0
        {'subm':f'b4.csv','short':f'b4'}, # 1
        {'subm':f'c4.csv','short':f'c4'}, # 2
        {'subm':f'd5.csv','short':f'd5'}, # 3
        {'subm':f'e5.csv','short':f'e5'}, # 4
        {'subm':f'f5.csv','short':f'f5'}, # 5
    ] 

    I = [0,1,2,3,4,5]

    subms  = [pd.read_csv(pds+f['subm']).rename(columns={'class':f['short']}) for f in ddf]
    shorts = [                                                   f['short']   for f in ddf]
    voters = [{'subm':subms[i],'short':shorts[i]} for i in I]
    
    dfs    = pd.merge(pd.merge(pd.merge(pd.merge(pd.merge(subms[I[0]],
                                                          subms[I[1]],on='id'),
                                                          subms[I[2]],on='id'),
                                                          subms[I[3]],on='id'),
                                                          subms[I[4]],on='id'),
                                                          subms[I[5]],on='id')
    def comp(x):
        _mask = x[voters[0]['short']]
        rezult = all([x[vtr['short']]==_mask for vtr in voters])
        return '=' if rezult==True else '!='
    
    dfs['comp'] = dfs.apply(lambda x: comp(x), axis=1)
    #---------------------------------------------------
    display(dfs.head(7)) ;print(dfs['comp'].value_counts(),'\n')
    #-----------------------------------------------------
    
    def vote(x):
        sh = [vtr['short'] for vtr in voters]
        if x[sh[0]]==x[sh[1]]==x[sh[2]]==x[sh[3]]==x[sh[4]]          : return x[sh[4]]
        if x[sh[0]]==x[sh[1]]==x[sh[2]]==x[sh[3]]          ==x[sh[5]]: return x[sh[5]]
        if x[sh[0]]==x[sh[1]]==x[sh[2]]          ==x[sh[4]]==x[sh[5]]: return x[sh[5]]
        if x[sh[0]]==x[sh[1]]          ==x[sh[3]]==x[sh[4]]==x[sh[5]]: return x[sh[5]]
        if x[sh[0]]          ==x[sh[2]]==x[sh[3]]==x[sh[4]]==x[sh[5]]: return x[sh[5]]
        if           x[sh[1]]==x[sh[2]]==x[sh[3]]==x[sh[4]]==x[sh[5]]: return x[sh[5]]
        return x[sh[5]]
    
    dfs['vote'] = dfs.apply(lambda x: vote(x), axis=1)
    #----------------------------------------------------
    display(dfs.head(7)) ;print(dfs['vote'].value_counts(),'\n')
    #------------------------------------------------------
    
    def vs(x,participant): return '=' if x['vote']==x[participant] else '!='
    
    sh = [vtr['short'] for vtr in voters]
    
    dfs[f'vs {sh[0]}'] = dfs.apply(lambda x:vs(x,sh[0]),axis=1) 
    dfs[f'vs {sh[1]}'] = dfs.apply(lambda x:vs(x,sh[1]),axis=1)
    dfs[f'vs {sh[2]}'] = dfs.apply(lambda x:vs(x,sh[2]),axis=1)
    dfs[f'vs {sh[3]}'] = dfs.apply(lambda x:vs(x,sh[3]),axis=1)
    dfs[f'vs {sh[4]}'] = dfs.apply(lambda x:vs(x,sh[4]),axis=1)
    dfs[f'vs {sh[5]}'] = dfs.apply(lambda x:vs(x,sh[5]),axis=1) 
    #----------------------------------------------------
    display(dfs.head(7)) 
    #------------------------------------------------------
    print(dfs[f'vs {sh[0]}'].value_counts(),'\n')
    print(dfs[f'vs {sh[1]}'].value_counts(),'\n')
    print(dfs[f'vs {sh[2]}'].value_counts(),'\n')
    print(dfs[f'vs {sh[3]}'].value_counts(),'\n')
    print(dfs[f'vs {sh[4]}'].value_counts(),'\n')
    print(dfs[f'vs {sh[5]}'].value_counts(),'\n')

    for file in 'a4,b4,c4,d5,e5,f5'.split(','):
        if os.path.isfile(file+'.csv'): os.remove(file+'.csv')

In [ ]:
if len(I) == 7:
    df  = pd.read_csv(sample_submission)

    def save(df,name_csv):
        df.rename(columns={'vote':'class'},inplace=True)
        df[['id','class']].to_csv(name_csv,index=False)

    df50 = vote5(I=I[0]) ;save(df50,'50.csv')
    df51 = vote5(I=I[1]) ;save(df51,'51.csv')
    df52 = vote5(I=I[2]) ;save(df52,'52.csv')
    df53 = vote5(I=I[3]) ;save(df53,'53.csv')
    df54 = vote5(I=I[4]) ;save(df54,'54.csv')
    df55 = vote5(I=I[5]) ;save(df55,'55.csv')
    df56 = vote5(I=I[6]) ;save(df56,'56.csv')

    pds = '/kaggle/working/'

    ddf = [
        {'subm':f'50.csv','short':f'50'}, # 0
        {'subm':f'51.csv','short':f'51'}, # 1
        {'subm':f'52.csv','short':f'52'}, # 2
        {'subm':f'53.csv','short':f'53'}, # 3
        {'subm':f'54.csv','short':f'54'}, # 4
        {'subm':f'55.csv','short':f'55'}, # 5
        {'subm':f'56.csv','short':f'56'}, # 6
    ] 

    I = [0,1,2,3,4,5,6]

    subms  = [pd.read_csv(pds+f['subm']).rename(columns={'class':f['short']}) for f in ddf]
    shorts = [                                                   f['short']   for f in ddf]
    voters = [{'subm':subms[i],'short':shorts[i]} for i in I]
    
    dfs    = pd.merge(pd.merge(pd.merge(pd.merge(pd.merge(pd.merge(subms[I[0]],
                                                                   subms[I[1]],on='id'),
                                                                   subms[I[2]],on='id'),
                                                                   subms[I[3]],on='id'),
                                                                   subms[I[4]],on='id'),
                                                                   subms[I[5]],on='id'),
                                                                   subms[I[6]],on='id')
    def comp(x):
        _mask = x[voters[0]['short']]
        rezult = all([x[vtr['short']]==_mask for vtr in voters])
        return '=' if rezult==True else '!='
    
    dfs['comp'] = dfs.apply(lambda x: comp(x), axis=1)
    #---------------------------------------------------
    display(dfs.head(7)) ;print(dfs['comp'].value_counts(),'\n')
    #-----------------------------------------------------
    
    def vote(x):
        s = [vtr['short'] for vtr in voters]
        if x[s[0]]==x[s[1]]==x[s[2]]==x[s[3]]==x[s[4]]==x[s[5]]         : return x[s[0]]
        if x[s[0]]==x[s[1]]==x[s[2]]==x[s[3]]==x[s[4]]         ==x[s[6]]: return x[s[0]]
        if x[s[0]]==x[s[1]]==x[s[2]]==x[s[3]]         ==x[s[5]]==x[s[6]]: return x[s[0]]
        if x[s[0]]==x[s[1]]==x[s[2]]         ==x[s[4]]==x[s[5]]==x[s[6]]: return x[s[0]]
        if x[s[0]]==x[s[1]]         ==x[s[3]]==x[s[4]]==x[s[5]]==x[s[6]]: return x[s[0]]
        if x[s[0]]         ==x[s[2]]==x[s[3]]==x[s[4]]==x[s[5]]==x[s[6]]: return x[s[2]]
        if          x[s[1]]==x[s[2]]==x[s[3]]==x[s[4]]==x[s[5]]==x[s[6]]: return x[s[1]]
        return x[s[6]]
    
    dfs['vote'] = dfs.apply(lambda x: vote(x), axis=1)
    #----------------------------------------------------
    display(dfs.head(7)) ;print(dfs['vote'].value_counts(),'\n')
    #------------------------------------------------------
    
    def vs(x,participant): return '=' if x['vote']==x[participant] else '!='
    
    sh = [vtr['short'] for vtr in voters]
    
    dfs[f'vs {sh[0]}'] = dfs.apply(lambda x:vs(x,sh[0]),axis=1) 
    dfs[f'vs {sh[1]}'] = dfs.apply(lambda x:vs(x,sh[1]),axis=1)
    dfs[f'vs {sh[2]}'] = dfs.apply(lambda x:vs(x,sh[2]),axis=1)
    dfs[f'vs {sh[3]}'] = dfs.apply(lambda x:vs(x,sh[3]),axis=1)
    dfs[f'vs {sh[4]}'] = dfs.apply(lambda x:vs(x,sh[4]),axis=1)
    dfs[f'vs {sh[5]}'] = dfs.apply(lambda x:vs(x,sh[5]),axis=1)
    dfs[f'vs {sh[6]}'] = dfs.apply(lambda x:vs(x,sh[6]),axis=1) 
    #----------------------------------------------------
    display(dfs.head(7)) 
    #------------------------------------------------------
    print(dfs[f'vs {sh[0]}'].value_counts(),'\n')
    print(dfs[f'vs {sh[1]}'].value_counts(),'\n')
    print(dfs[f'vs {sh[2]}'].value_counts(),'\n')
    print(dfs[f'vs {sh[3]}'].value_counts(),'\n')
    print(dfs[f'vs {sh[4]}'].value_counts(),'\n')
    print(dfs[f'vs {sh[5]}'].value_counts(),'\n')
    print(dfs[f'vs {sh[6]}'].value_counts(),'\n')

    for file in '50,51,52,53,54,55,56'.split(','):
        if os.path.isfile(file+'.csv'): os.remove(file+'.csv')

## Submit

In [ ]:
df = pd.read_csv(sample_submission)

df['class'] = dfs['vote']

df.to_csv('submission.csv',index=False)
df